In [11]:
#imports
import numpy as np 
import pandas as pd
import time 
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [17]:
#upload file
#/home/acoutant/Nextcloud_CNRS/Recherche/Science History, Environment and Social Sciences/Environmental sciences and ecology/Fossil Fuels & Banks/Data ESR fundings/Clean Data/'


In [2]:
###############################################################
#
#             EU fundings - EXTRACT AND VIZUALIZE RESULTS
#             From programmes HORIZON and HORIZON 2020 
#
###############################################################
#                       TO DO LIST: 
#               1) 
###############################################################

In [16]:
Start = time.time()
##################### GET CLEAN DATA #####################
#CleanFolderName = '/home/acoutant/Nextcloud_CNRS/Recherche/Science History, Environment and Social Sciences/Environmental sciences and ecology/Fossil Fuels & Banks/Data ESR fundings/Clean Data/'
CleanFolderName = '/Users/annalewicki/Nextcloud/Programmes/1-Réorienter la recherche/Politique de la recherche/Europe/Préparation FP10/cordis-HORIZONprojects/'
Clean_file = 'project.xlsx'
#Clean_file = 'EU-fundings.csv'
# Clean_file = 'EU-fundings - full EU zone.csv'

EUfullTable = pd.read_csv(CleanFolderName + Clean_file,dtype = {'ecContribution': float,'Source': int}) 
EUfullTable['EuroSciVoc'] = EUfullTable['EuroSciVoc'].fillna('[]').apply(lambda x: eval(x))

# EUfullTable = EUfullTable[EUfullTable['source'] < 6] # TO SELECT A SUBSET OF THE TABLE 

AcSeries = EUfullTable['projectAcronym']
SDateSeries = EUfullTable['Start Date']
LabsAndIFSeries = EUfullTable['name']
IFmainSeries = EUfullTable['Fossil Fuel Company']
FundSeries = EUfullTable['ecContribution']
CountrySeries = EUfullTable['country']

Nt = AcSeries.size # Total line number 

##################### GET IF LIST #####################
IFlistFolderName = '/home/acoutant/Nextcloud_CNRS/Recherche/Science History, Environment and Social Sciences/Environmental sciences and ecology/Fossil Fuels & Banks/Data ESR fundings/Super Data IF/'
IFlist_file = 'Super List IF.csv'

IFtable = pd.read_csv(IFlistFolderName + IFlist_file) 
IF_list_full = list(IFtable['Entreprise Fossile (Nom, autre/ancien nom, ou filiale)'])
IF_list = list(IFtable['Entreprise parent'])
IF_dico = dict(zip(IF_list_full,IF_list)) # Dico from full name to main IF name
"""
##################### TIME SERIES ##################### 
"""

TimeTable = EUfullTable[['projectAcronym','Start Date']]
TimeTable = TimeTable.drop_duplicates(subset=['projectAcronym'])
TimeTable['Start Date'] = TimeTable['Start Date'].str[0:4] # Select year

Nproj = TimeTable['Start Date'].value_counts() 
Nproj = Nproj.apply(int).sort_index()
Time = Nproj.keys().astype(int)

Nf = sum(Nproj)

"""
##################### MONEY FOR IF #####################
"""
IFfullfundsCols = ['Fossil Fuel Company','ecContribution'] 
IFfullfunds_table = EUfullTable[IFfullfundsCols].loc[IFmainSeries != '-']

IFfunds_table = IFfullfunds_table.groupby('Fossil Fuel Company').sum() # !! groupby makes a weird table...
IFfundsM_series = IFfunds_table['ecContribution'].map(lambda x: x/1e6)

IFprojNum_series = IFfullfunds_table['Fossil Fuel Company'].value_counts()

##################### ORDER BY FUNDING #####################

IFfundsM_series = IFfundsM_series.sort_values(ascending=False)
IFprojNum_series = IFprojNum_series.sort_values(ascending=False)

Ntot = IFfundsM_series.size 
NIFshow = 15 # Show the richest only
IFfundsM_Rseries = IFfundsM_series.iloc[0:NIFshow] 
IFprojNum_Rseries = IFprojNum_series.iloc[0:NIFshow] 

"""
##################### MONEY FOR FR LABS #####################
"""
LabsFRfull_table = EUfullTable[['name','ecContribution']].loc[CountrySeries == 'FR']
LabsFRfull_table = LabsFRfull_table.loc[IFmainSeries == '-']

LabsFR_table = LabsFRfull_table.groupby('name').sum() 
LabsFRfundsM_series = LabsFR_table['ecContribution'].map(lambda x: x/1e6)

LabsFRprojNum_series = LabsFRfull_table['name'].value_counts()

##################### RANKED SERIES #####################

LabsFRfundsM_series = LabsFRfundsM_series.sort_values(ascending=False)
LabsFRprojNum_series = LabsFRprojNum_series.sort_values(ascending=False) 

NlabShow = 15 
LabsFRfundsM_Rseries = LabsFRfundsM_series[0:NlabShow] 
LabsFRprojNum_Rseries = LabsFRprojNum_series[0:NlabShow] 

"""
##################### MONEY FOR ALL #####################
"""
Allfull_table = EUfullTable[['name','ecContribution']]

All_table = Allfull_table.groupby('name').sum()
All_series = All_table['ecContribution'].map(lambda x: x/1e6) 

All_series = All_series.sort_values()

"""
##################### KEYWORDS STATS #####################
"""
KeyWords_Nlist = list(EUfullTable.drop_duplicates(subset=['projectAcronym'])['EuroSciVoc']) # gives a nested list 
KeyWords_list = [item for sublist in KeyWords_Nlist for item in sublist] # Flatten the nested list 
KeyWords_set = set(KeyWords_list)

KeyWords_Count = pd.Series(KeyWords_list).value_counts() 
Nkw = 25 # Number of keywords to show 

KeyWords_RCount = KeyWords_Count.iloc[0:Nkw] 

End = time.time()
print('Elapsed time: '+str(End-Start))
#%%

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb9 in position 10: invalid start byte

In [4]:
###############################################################
###############################################################
###############################################################
#
#                           PLOTS 
#
###############################################################
###############################################################
###############################################################

In [5]:
Font=14
Line=2
Marker=2 
plt.rc('text', usetex=False) # FORCE NO LATEX
plt.rc('font',size=Font)
plt.rc('font',family='serif')
#####################
#   Time series  
##################### 
plt.rc('ytick', labelsize=10) 
plt.rc('xtick', labelsize=10)  
plt.figure() 
plt.bar(Time,Nproj,color = 'tab:olive',zorder=3) 
plt.grid()
if Time.size > 14: 
    plt.xticks(Time[0]+2*np.arange(1+int(Time.size/2)))
plt.title(str(Nf)+' projets involving fossil fuel companies')

plt.show()
#%%
#####################
#   IFs funding
##################### 
plt.rc('ytick', labelsize=12) 
plt.rc('xtick', labelsize=16)  
plt.figure() 
plt.barh(IFfundsM_Rseries.index,IFfundsM_Rseries,zorder=3) 
plt.grid()
plt.title('Total funding obtained')
plt.xlabel('M €')

#####################
# IFs number of projects 
##################### 
plt.figure() 
plt.barh(IFprojNum_Rseries.index,IFprojNum_Rseries,color = 'tab:purple',zorder=3) 
plt.grid()
plt.title('Total number of projects obtained')
plt.xlabel('Number of projects')

plt.show()
#%%
#####################
#   Labs funding
##################### 
plt.rc('ytick', labelsize=12) 
plt.rc('xtick', labelsize=16) 
plt.figure() 
plt.barh(LabsFRfundsM_Rseries.index,LabsFRfundsM_Rseries,zorder=3) 
plt.grid()
plt.title('Total funding obtained')
plt.xlabel('M €')

#####################
#   Labs number of proj. 
##################### 
plt.figure() 
plt.barh(LabsFRprojNum_Rseries.index,LabsFRprojNum_Rseries,zorder=3) 
plt.grid()
plt.title('Total number of projects obtained')
plt.xlabel('Number of projects')

plt.show()
#%%
#####################
#   Keywords count 
##################### 
plt.rc('ytick', labelsize=10) 
plt.rc('xtick', labelsize=16) 
plt.figure() 
plt.barh(KeyWords_RCount.index,KeyWords_RCount,zorder=3) 
plt.grid()
plt.title('Most used keywords (EuroSciVoc)')
plt.xlabel('Number of projects with that keyword')

plt.show()

NameError: name 'plt' is not defined